# La dette ne se rembourse pas toujours : parfois, elle fond · *Debt is not always repaid: sometimes it melts*

Notebook compagnon du chapitre **10. PIB réel et PIB nominal : pourquoi corriger de l'inflation** — [lire l'article](https://nmlab.io/ressources/pib-reel-pib-nominal).
Companion notebook to chapter **10. Real GDP and Nominal GDP: Why Correct for Inflation** — [read the article](https://nmlab.io/en/ressources/real-gdp-nominal-gdp).

**Exécutez l'unique cellule ci-dessous** (bouton ▶ ou Ctrl+Entrée) : la figure se régénère avec les **données FRED du jour**. Passez `LANG = "en"` en tête de cellule pour les libellés anglais. — Run the single cell below (▶ or Ctrl+Enter) to rebuild the figure with **today's FRED data**; set `LANG = "en"` at the top for English labels.

Code : licence MIT · © 2026 [NMLab](https://nmlab.io) · dépôt [nmlab-finance/nmlab-figures](https://github.com/nmlab-finance/nmlab-figures)

In [ ]:
LANG = "fr"   # "fr" ou "en" — langue des libellés / label language

# Récupère puis active le style partagé NMLab (thème sombre + police Inter).
# Fetch and activate the shared NMLab style (dark theme + Inter font).
import urllib.request

urllib.request.urlretrieve("https://raw.githubusercontent.com/nmlab-finance/nmlab-figures/main/nmlab_style.py", "nmlab_style.py")
import nmlab_style as nm

nm.setup()


from pandas import Series

def load_debt() -> Series:
    """Dette fédérale américaine brute en % du PIB (GFDGDPA188S), série annuelle depuis 1939,
    en direct depuis FRED.
    U.S. gross federal debt as % of GDP, annual since 1939, live from FRED."""
    return nm.load_fred("GFDGDPA188S")

debt = load_debt()


import matplotlib.dates as mdates
from matplotlib.figure import Figure

LABELS = {
    "fr": dict(
        title="La dette ne se rembourse pas toujours : parfois, elle fond",
        sub="Dette fédérale américaine en % du PIB — un ratio nominal sur un PIB nominal",
        ylab="dette fédérale brute, % du PIB",
        ann_1946="1946 : 119 %", ann_1974="1974 : 31 %", ann_2025="2025 : 121 %",
        mid="divisée par quatre en trente ans,\nsans grands excédents : le PIB\nnominal courait plus vite",
        note="Le numérateur (la dette) est figé en dollars d'époque ; le dénominateur (PIB nominal) gonfle\n"
             "avec les volumes ET les prix. L'inflation rembourse en silence. Source : OMB via FRED."),
    "en": dict(
        title="Debt is not always repaid: sometimes it melts",
        sub="U.S. federal debt as % of GDP — a nominal ratio over a nominal GDP",
        ylab="gross federal debt, % of GDP",
        ann_1946="1946: 119%", ann_1974="1974: 31%", ann_2025="2025: 121%",
        mid="cut by four in thirty years,\nwith no big surpluses: nominal\nGDP was running faster",
        note="The numerator (the debt) is frozen in the dollars of its era; the denominator (nominal GDP)\n"
             "swells with volumes AND prices. Inflation repays in silence. Source: OMB via FRED."),
}

def _at(series: Series, year: int):
    mask = series.index.year == year
    return series.index[mask][0], float(series[mask].iloc[0])

def build_figure(debt: Series, lang: str) -> Figure:
    """Aire remplie du ratio dette/PIB, 1939-2025, avec les jalons 1946, 1974 et 2025."""
    text = LABELS[lang]
    fig = nm.figure(height_px=1140)
    ax = nm.axes(fig, left=0.078)
    ax.fill_between(debt.index, debt, color=nm.COLORS["blue"], alpha=0.16, zorder=1)
    ax.plot(debt.index, debt, color=nm.COLORS["blue"], lw=3.2, zorder=3)
    ax.set_ylim(0, 145)
    ax.set_yticks(range(0, 141, 20))
    ax.set_ylabel(text["ylab"])
    ax.margins(x=0.02)
    ax.xaxis.set_major_locator(mdates.YearLocator(10))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    d46, v46 = _at(debt, 1946)
    ax.scatter([d46], [v46], s=130, color=nm.COLORS["text"], zorder=5)
    ax.annotate(text["ann_1946"], xy=(d46, v46), xytext=(58, 14), textcoords="offset points",
                ha="left", va="center", fontsize=24, fontweight="bold", color=nm.COLORS["text"],
                arrowprops=dict(arrowstyle="-", color=nm.COLORS["muted"], lw=1.3))
    d74, v74 = _at(debt, 1974)
    ax.scatter([d74], [v74], s=130, color=nm.COLORS["text"], zorder=5)
    ax.annotate(text["ann_1974"], xy=(d74, v74), xytext=(0, -78), textcoords="offset points",
                ha="center", va="center", fontsize=24, fontweight="bold", color=nm.COLORS["text"],
                arrowprops=dict(arrowstyle="-", color=nm.COLORS["muted"], lw=1.3))
    d25, v25 = _at(debt, 2025)
    ax.scatter([d25], [v25], s=130, color=nm.COLORS["text"], zorder=5)
    ax.annotate(text["ann_2025"], xy=(d25, v25), xytext=(-30, 46), textcoords="offset points",
                ha="center", va="center", fontsize=24, fontweight="bold", color=nm.COLORS["text"],
                arrowprops=dict(arrowstyle="-", color=nm.COLORS["muted"], lw=1.3))

    ax.text(0.30, 0.44, text["mid"], transform=ax.transAxes, ha="center", va="center",
            fontsize=23, color=nm.COLORS["text"], linespacing=1.5)
    nm.header(fig, text["title"], text["sub"])
    nm.footer(fig, text["note"])
    return fig

build_figure(debt, LANG)